# Milestone 2: Visualize & Explore
**DATA 101 – Term Project** · Miguel Angel Albarracin Ospina

**Research question:** *How do body weight and age affect the amount of weight powerlifters can lift?*

**Dataset:** [OpenPowerlifting](https://www.openpowerlifting.org/data.html) (open data from powerlifting meets around the world)

In this notebook I use summary statistics, tables and charts to look for patterns that help answer my question.

## 1. Get the data and load the clean version
First the notebook downloads the OpenPowerlifting file (only if it is not here yet). Then it repeats the same cleaning steps from Milestone 1, in a shorter way, so this notebook can run by itself.

One small change: I now use **5-year age groups** (15-19, 20-24, ...), because they are easier to read than my first ones.

In [ ]:
import glob, os, shutil, urllib.request, zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

URL = "https://openpowerlifting.gitlab.io/opl-csv/files/openpowerlifting-latest.zip"

if not glob.glob("openpowerlifting-*.csv"):
    urllib.request.urlretrieve(URL, "openpowerlifting-latest.zip")
    with zipfile.ZipFile("openpowerlifting-latest.zip") as z:
        name = next(n for n in z.namelist() if n.endswith(".csv"))
        with z.open(name) as src, open(os.path.basename(name), "wb") as dst:
            shutil.copyfileobj(src, dst)

DATA_PATH = sorted(glob.glob("openpowerlifting-*.csv"))[0]
print("Reading:", DATA_PATH)

In [ ]:
# ---- Cleaning (same steps as Milestone 1) ----
cols = ["Name", "Sex", "Event", "Equipment", "Age", "BodyweightKg",
        "Best3SquatKg", "Best3BenchKg", "Best3DeadliftKg", "TotalKg",
        "Place", "Tested", "Federation", "Date", "MeetName"]
raw = pd.read_csv(DATA_PATH, usecols=cols, low_memory=False)

df = raw.copy()
for c in ["Name", "Sex", "Event", "Equipment", "Place", "Tested", "Federation", "MeetName"]:
    df[c] = df[c].astype("string").str.strip()
df["Sex"] = df["Sex"].str.upper()
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

df = df[df["Sex"].isin(["M", "F"]) & (df["Event"] == "SBD") & (df["Equipment"] == "Raw")]
df = df[df["Place"].fillna("").ne("DQ")]

lifts = ["Best3SquatKg", "Best3BenchKg", "Best3DeadliftKg", "TotalKg"]
df = df.dropna(subset=["Age", "BodyweightKg"] + lifts)
df = df[df["Age"].between(14, 90) & df["BodyweightKg"].between(35, 250) & (df[lifts] > 0).all(axis=1)]
df = df.drop_duplicates(subset=["Name", "Date", "MeetName", "Federation", "Sex", "Age", "BodyweightKg", "TotalKg"])

# ---- New columns ----
age_bins = list(range(10, 60, 5)) + [60, 100]
age_labels = [f"{a}-{a+4}" for a in range(10, 60, 5)] + ["60+"]
df["AgeGroup"] = pd.cut(df["Age"], bins=age_bins, labels=age_labels, right=False)

bw_bins = [35, 60, 70, 80, 90, 100, 110, 125, 251]
bw_labels = ["35-59", "60-69", "70-79", "80-89", "90-99", "100-109", "110-124", "125+"]
df["BodyweightGroup"] = pd.cut(df["BodyweightKg"], bins=bw_bins, labels=bw_labels, right=False)

df["TotalToBodyweight"] = df["TotalKg"] / df["BodyweightKg"]
df["IsTested"] = df["Tested"].fillna("No").eq("Yes")
df = df.reset_index(drop=True)

print(f"Clean data: {len(df):,} rows (from {len(raw):,} raw rows)")

In [ ]:
# Plot settings: same colors for men and women in every chart
COLORS = {"M": "#0072B2", "F": "#E69F00"}
NAMES = {"M": "Men", "F": "Women"}
plt.rcParams.update({
    "figure.figsize": (8, 4.5), "axes.grid": True, "axes.grid.axis": "y",
    "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False,
})

## 2. Summary statistics
First the basic numbers: the average, the middle value (median), the spread and the extremes of each number column.

In [ ]:
num_cols = ["Age", "BodyweightKg", "Best3SquatKg", "Best3BenchKg", "Best3DeadliftKg", "TotalKg", "TotalToBodyweight"]
df[num_cols].describe().T

In [ ]:
# The same numbers, split by sex (men and women are very different, so I keep them apart)
df.groupby("Sex", observed=True)[["Age", "BodyweightKg", "TotalKg", "TotalToBodyweight"]].agg(["count", "mean", "median", "std"]).round(2)

## 3. Aggregated tables
These tables group the results by age and by body weight. The **median** is better than the average here, because a few very strong lifters pull the average up.

In [ ]:
# Median total (kg) by age group
age_table = df.pivot_table(index="AgeGroup", columns="Sex", values="TotalKg", aggfunc=["median", "count"], observed=False).round(1)
age_table

In [ ]:
# Median total (kg) by body weight group
bw_table = df.pivot_table(index="BodyweightGroup", columns="Sex", values="TotalKg", aggfunc=["median", "count"], observed=False).round(1)
bw_table

In [ ]:
# Strength for body size: total divided by body weight, by age group
df.pivot_table(index="AgeGroup", columns="Sex", values="TotalToBodyweight", aggfunc="median", observed=False).round(2)

## 4. Univariate charts (one variable at a time)
These show what each variable looks like on its own: who the lifters are and how much they lift.

In [ ]:
# Age of the lifters
fig, ax = plt.subplots()
ax.hist(df["Age"], bins=np.arange(14, 80, 2), color="#56B4E9", edgecolor="white")
ax.set_title("Age of lifters")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Number of results")
plt.show()

In [ ]:
# Body weight, men vs women
fig, ax = plt.subplots()
for s in ["M", "F"]:
    ax.hist(df.loc[df["Sex"] == s, "BodyweightKg"], bins=50, alpha=0.65, color=COLORS[s], label=NAMES[s])
ax.set_title("Body weight of lifters")
ax.set_xlabel("Body weight (kg)")
ax.set_ylabel("Number of results")
ax.legend()
plt.show()

In [ ]:
# Total lifted, men vs women
fig, ax = plt.subplots()
for s in ["M", "F"]:
    ax.hist(df.loc[df["Sex"] == s, "TotalKg"], bins=60, alpha=0.65, color=COLORS[s], label=NAMES[s])
ax.set_title("Total lifted (squat + bench + deadlift)")
ax.set_xlabel("Total (kg)")
ax.set_ylabel("Number of results")
ax.legend()
plt.show()

In [ ]:
# How many results do we have in each age group? (a bar chart)
counts = df["AgeGroup"].value_counts().sort_index()
fig, ax = plt.subplots()
ax.bar(counts.index.astype(str), counts.values, color="#56B4E9")
ax.set_title("Number of results by age group")
ax.set_xlabel("Age group")
ax.set_ylabel("Number of results")
plt.xticks(rotation=45)
plt.show()

**What these charts show**
- The number of results is not the same in every age group. Some groups (the youngest and the oldest) have fewer results, so they need more care.
- Men and women have different body weights and totals, so I keep them in separate colors in every chart.

## 5. Multivariate charts (two or more variables together)
Here I look at my real question: how age and body weight are related to the total lifted.

In [ ]:
# Body weight vs total (random sample so the chart stays readable)
sample = df.sample(n=min(20000, len(df)), random_state=42)
fig, ax = plt.subplots()
for s in ["M", "F"]:
    part = sample[sample["Sex"] == s]
    ax.scatter(part["BodyweightKg"], part["TotalKg"], s=5, alpha=0.2, color=COLORS[s], label=NAMES[s])
    slope, intercept = np.polyfit(part["BodyweightKg"], part["TotalKg"], 1)
    xs = np.array([part["BodyweightKg"].min(), part["BodyweightKg"].max()])
    ax.plot(xs, slope * xs + intercept, color="black", linewidth=1.5)
ax.set_title("Body weight vs total lifted (black lines = trend)")
ax.set_xlabel("Body weight (kg)")
ax.set_ylabel("Total (kg)")
ax.legend(markerscale=4)
plt.show()

In [ ]:
# Median total by age group (line chart)
med = df.groupby(["AgeGroup", "Sex"], observed=True)["TotalKg"].median().unstack("Sex")
fig, ax = plt.subplots()
for s in ["M", "F"]:
    ax.plot(med.index.astype(str), med[s], marker="o", color=COLORS[s], label=NAMES[s])
ax.set_title("Median total by age group")
ax.set_xlabel("Age group")
ax.set_ylabel("Median total (kg)")
ax.legend()
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Strength for body size (total / body weight) by age group (bar chart)
ratio = df.groupby(["AgeGroup", "Sex"], observed=True)["TotalToBodyweight"].median().unstack("Sex")
x = np.arange(len(ratio))
fig, ax = plt.subplots()
ax.bar(x - 0.2, ratio["M"], width=0.4, color=COLORS["M"], label="Men")
ax.bar(x + 0.2, ratio["F"], width=0.4, color=COLORS["F"], label="Women")
ax.set_xticks(x)
ax.set_xticklabels(ratio.index.astype(str), rotation=45)
ax.set_title("Total lifted for each kg of body weight, by age group")
ax.set_xlabel("Age group")
ax.set_ylabel("Median total / body weight")
ax.legend()
plt.show()

In [ ]:
# Spread of the total for each body weight group (box plot, men)
men = df[df["Sex"] == "M"]
groups = [g for g in bw_labels if (men["BodyweightGroup"] == g).sum() >= 30]
data = [men.loc[men["BodyweightGroup"] == g, "TotalKg"] for g in groups]
fig, ax = plt.subplots()
ax.boxplot(data, tick_labels=groups, showfliers=False, patch_artist=True,
           boxprops=dict(facecolor="#9CC9E6"), medianprops=dict(color="black"))
ax.set_title("Total lifted by body weight group (men)")
ax.set_xlabel("Body weight group (kg)")
ax.set_ylabel("Total (kg)")
plt.show()

In [ ]:
# Age AND body weight together: median total for each combination (heatmap)
# Combinations with fewer than 30 results are left blank because they are not reliable.
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, s in zip(axes, ["M", "F"]):
    part = df[df["Sex"] == s]
    pt = part.pivot_table(index="BodyweightGroup", columns="AgeGroup", values="TotalKg", aggfunc="median", observed=False)
    cnt = part.pivot_table(index="BodyweightGroup", columns="AgeGroup", values="TotalKg", aggfunc="count", observed=False)
    pt = pt.where(cnt >= 30)
    im = ax.imshow(pt.values, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(pt.columns)))
    ax.set_xticklabels(pt.columns.astype(str), rotation=45)
    ax.set_yticks(range(len(pt.index)))
    ax.set_yticklabels(pt.index.astype(str))
    ax.set_title(f"Median total (kg) - {NAMES[s]}")
    ax.set_xlabel("Age group")
    ax.grid(False)
    fig.colorbar(im, ax=ax)
axes[0].set_ylabel("Body weight group (kg)")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with the total (it shows how strong a straight-line relationship is, not the cause)
for s in ["M", "F"]:
    print(f"{NAMES[s]}:")
    print(df.loc[df["Sex"] == s, ["Age", "BodyweightKg", "TotalKg"]].corr()["TotalKg"].drop("TotalKg").round(2).to_string(), "\n")

In [ ]:
# Where would I land? (my sex, age group and body weight group)
MY_SEX = "M"
MY_AGE_GROUP = "15-19"
MY_BW_GROUP = "70-79"   # I weigh about 73 kg

mine = df[(df["Sex"] == MY_SEX) & (df["AgeGroup"] == MY_AGE_GROUP) & (df["BodyweightGroup"] == MY_BW_GROUP)]
print(f"Lifters in my group: {len(mine):,}")
if len(mine):
    print(mine[["Best3SquatKg", "Best3BenchKg", "Best3DeadliftKg", "TotalKg"]].median().round(1).to_string())

## 6. Key insights
The numbers in this text are calculated from the data above.

In [ ]:
corr_bw = {s: df.loc[df["Sex"] == s, ["BodyweightKg", "TotalKg"]].corr().iloc[0, 1] for s in ["M", "F"]}
age_med = df.groupby(["Sex", "AgeGroup"], observed=True)["TotalKg"].median()
peak_total = {s: age_med[s].idxmax() for s in ["M", "F"]}
ratio_med = df.groupby(["Sex", "AgeGroup"], observed=True)["TotalToBodyweight"].median()
peak_ratio = {s: ratio_med[s].idxmax() for s in ["M", "F"]}
med_total = df.groupby("Sex", observed=True)["TotalKg"].median()

men_bw = df[df["Sex"] == "M"].groupby("BodyweightGroup", observed=True)["TotalKg"].median().dropna()
n_age = df["AgeGroup"].value_counts()
n_age = n_age[n_age > 0]
most_common_age = n_age.idxmax()
least_common_age = n_age.idxmin()

display(Markdown(f"""
1. **Body weight and total go together.** The correlation is **{corr_bw['M']:.2f}** for men and **{corr_bw['F']:.2f}** for women. In the scatterplot the trend lines go up, so heavier lifters tend to lift more.
2. **The difference between body weight groups is big.** For men, the median total goes from **{men_bw.min():.0f} kg** (group {men_bw.idxmin()} kg) to **{men_bw.max():.0f} kg** (group {men_bw.idxmax()} kg).
3. **Age follows a different pattern than body weight.** The age group with the highest median total is **{peak_total['M']}** for men and **{peak_total['F']}** for women. When I divide the total by body weight, the top group is **{peak_ratio['M']}** for men and **{peak_ratio['F']}** for women.
4. **Men and women need separate charts.** The median total is **{med_total['M']:.0f} kg** for men and **{med_total['F']:.0f} kg** for women.
5. **The data is not equal in every group.** The most common age group is **{most_common_age}** (**{n_age.max():,}** results) and the least common is **{least_common_age}** (**{n_age.min():,}** results), so the small groups are less reliable.

**How this refines my question:** age and body weight both matter, but body weight has the stronger straight-line link. A better question for Milestone 3 is: *"Can I predict a powerlifter's total from body weight, age and sex?"*
"""))

## 7. Reflection

### Which visualizations showed me something interesting?
The **heatmap** was the most useful for my question, because it is the only chart that shows age and body weight at the same time. My question has two things that might matter, so I needed one chart that puts them together. The **scatterplot** was second: it shows every lifter as a dot, so I can see how spread out the results are, and not only the average. The **line chart by age group** showed me that age does not work like body weight. Body weight pushes the total up in a clear way, but age follows a less simple pattern, with different results in each age group.

I also learned that the *type* of chart matters. A histogram is good to see one variable, a bar chart is good to compare groups, a scatterplot is good to see two numbers together, and a heatmap is good when I have two groups and one number. When I used the wrong chart, the pattern was harder to see.

### How do professionals in my field use data visuals?
I am studying psychology, and psychologists use charts all the time. Researchers use **histograms** to check how scores are spread before they run a test. They use **scatterplots** to see if two things are related (for example, hours of sleep and mood), and **bar charts** to compare groups, like a treatment group against a control group. Therapists and clinics can use **line charts** to follow a person's symptoms over time and see if something is getting better. In all of these, a good chart helps a person understand the result fast, and also helps other people check if the conclusion is honest.

It also made me think about being careful. A chart can make a pattern look stronger than it is, and a correlation does not prove a cause. For example, here I cannot say that gaining weight makes someone lift more, because things like training years and experience are not in the data.

### Limits to remember
- Lifters in meets are not average people, so these results do not describe everyone.
- The same lifter can appear many times, so the rows are not fully independent.
- Groups with few results (very young, very old, very light, very heavy) are less reliable.

### Next step
In Milestone 3 I will try a simple model to predict the total from body weight, age and sex, and check how good it is.